# SPATIAL INTELLIGENCE —THE EDGE_ Carrasco, Montevideo — Foster + Partners / Ponce de León Architects

Analyze **four stacked floor plans** of *The Edge* (Carrasco, Montevideo — Foster + Partners / Ponce de León Architects) as a single connected building and run the Spatial Intelligence workflows (Session 03 / Assignment 02) **across all floors simultaneously**.

**Geometry note.** The source `Edge_4-Floor-Plans.obj` contains four 2D plans: **Planta Baja (GF)**, **Piso 1 (F1)**, **Piso 2 (F2)**, and **Terraza/Rooftop**. After importing with `Topology.ByOBJPath` (the project's Z-up convention) the four plans lie flat in the **XY plane** at four discrete levels **Z = 0, 4, 8, 12**. Each plan is a slab containing only 8 exclusive residences (units 101–104 and 201–204) plus shared circulation; its walls and the **4 elevator shafts** (shared across the 8 residences, not one per unit) and stairs are the *holes* in the navigable mesh.

**Unit-to-floor mapping (from the project's floor/unit matrix):**
- **101** — Duplex, GF + F1
- **102** — Single floor, F1 only
- **103** — Single floor, F1 only
- **104** — Duplex, GF + F1
- **201** — Duplex, F2 + Rooftop
- **202** — Duplex, F2 + Rooftop
- **203** — Duplex, F2 + Rooftop
- **204** — Triplex, F1 + F2 + Rooftop

**Method (instructor's grid-sampling strategy).** For every floor we overlay a regular grid and keep the grid points that fall inside the navigable (meshed) area — interior rooms, terraces, private gardens, and the rooftop deck/jacuzzi areas. Those points become graph nodes; neighbouring valid points are joined by edges, producing one graph per floor.

**Stitching the floors.** The four floor graphs are merged pairwise with `Graph.Union` (e.g. `g3 = Graph.Union(g1, g2)`). Two kinds of cross-floor links are then added on top of the union:
- **Elevator links:** the 4 shared elevator shafts, located by finding the matching vertex coordinates stacked directly above one another on each floor.
- **Duplex/triplex internal-stair links:** for each duplex/triplex unit, the matching vertex pair `(v1, v2)` stacked on top of each other (its private internal staircase) is found and a manual edge is created with `e = Edge.ByVertices([v1, v2])` and added to the merged graph with `g3 = Graph.AddEdge(g3, e)`. Specifically:
  - 101: GF↔F1
  - 104: GF↔F1
  - 201: F2↔Rooftop
  - 202: F2↔Rooftop
  - 203: F2↔Rooftop
  - 204: F1↔F2 and F2↔Rooftop (two internal-stair edges, since it's a triplex)
  
  Units 102 and 103 get no extra edges since they don't cross floors — they connect to the rest of the building only through the 4 shared elevator nodes on F1.

This way the duplex/triplex units contribute **extra private cross-floor connections** in addition to the 4 shared elevator/stair cores, so that connectivity metrics such as **Degree Centrality are computed over the WHOLE building, not floor-by-floor.**

**Visual format.** Results are exported like this: each grid node becomes a small square cell coloured by the metric and rendered with `Topology.Show(faces, faceColorKey=...)` on a black background (filled heatmaps, not scatter points). The four floors are laid out one above the other in each heatmap.

Workflows included: Shortest Path (cross-floor, e.g. GF garden → Terraza jacuzzi, potentially routed through a duplex/triplex unit's private internal stair rather than the shared elevators), Closeness Centrality (Integration), Betweenness Centrality (Choice), Community Detection (testing whether each duplex/triplex unit emerges as its own spatial community, now reinforced by its private internal-stair edge — while 102/103 should remain tied to the shared F1 elevator core), building-wide Degree Centrality (expect the 4 shared elevator nodes plus the duplex/triplex internal-stair nodes to show as the high-degree cross-floor hubs), and per-floor Visibility/Isovist analysis (sea/garden view exposure from the floor-to-ceiling glazing).

## 1. Import the needed libraries

In [ ]:
import os, math, time
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Render every figure as a STATIC image via kaleido (no WebGL). This is what stops
# VS Code's "WebGL is not supported" crash: the interactive webview is never used.
pio.renderers.default = "png"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
renderer = "png"      # "png" renders via kaleido (no WebGL) — works in VS Code without GPU
                      # Change to "browser" or "notebook" if you want interactive figures

# Renderer used ONLY for the two 3D-graph cells you want to rotate/pan/zoom
# (building graph + shortest path). "browser" opens them in your default web browser,
# which is the most reliable way around VS Code's WebGL crash/freeze. Switch to
# "notebook" to embed them inline if your VS Code build handles WebGL well.
INTERACTIVE_RENDERER = "png"

# Paths
BASE_DIR  = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Final_Project"
OBJ_PATH  = os.path.join(BASE_DIR, "3D-Models", "Marsella_3-Floor-Plans.obj")
ASSETS_DIR = os.path.join(BASE_DIR, "Notebooks", "assets", "assets_normal_building_floorplans_1.0m-grid")
os.makedirs(ASSETS_DIR, exist_ok=True)

# Floor levels (Z value of each plan after import) ordered bottom -> top
FLOOR_LEVELS = [0, 4, 8]
FLOOR_NAMES  = ["Floor 1", "Floor 2", "Floor 3"]

# Analysis grid spacing (plan units). Smaller = finer & slower. 3.0 is a good balance.
GRID_SIZE = 1

# Vertical spacing used to STACK the floors in the combined 3D graph (visual only)
FLOOR_HEIGHT = 30

# ---------------------------------------------------------------------------
# Stair / vertical-circulation locations in PLAN coordinates (X, Y).
# find_closest_node uses only (X, Y); the Z each point was picked at just tells
# us which floors it links. Floor indices: 0 = Floor 1, 1 = Floor 2, 2 = Floor 3.
# ---------------------------------------------------------------------------

# Floor 3 <-> Floor 2  (pair (1, 2))   -- points picked at Z = 8
LEVEL_3_TO_2 = [
    (236.40093, 350.06035), (235.23132, 350.06035), (227.94559, 350.06035),
    (226.77598, 350.06035), (219.49025, 350.06035), (218.32064, 350.06035),
    (211.03491, 350.03151), (209.86530, 350.03151), (185.71313, 350.04576),
    (184.54352, 350.04576), (177.25779, 350.03151), (176.08818, 350.03151),
    (168.80245, 350.04576), (167.63284, 350.04576), (160.34711, 350.04576),
    (159.17750, 350.04576), (151.88509, 350.03151), (150.71548, 350.03151),
    (135.02587, 350.03151), (133.85626, 350.03151), (126.57053, 350.04900),
    (125.40092, 350.04900), (143.42306, 350.04576), (192.99470, 350.04738),
]

# Floor 2 <-> Floor 1  (pair (0, 1))   -- points picked at Z = 4
LEVEL_2_TO_1 = [
    (126.44623, 340.78751), (125.27662, 340.78751), (134.90157, 340.78751),
    (133.73196, 340.78751), (143.35691, 340.78751), (142.18730, 340.78751),
    (151.81225, 340.78751), (150.64264, 340.78751), (160.26759, 340.78751),
    (159.09798, 340.78751), (168.72294, 340.78751), (167.55332, 340.78751),
    (177.17828, 340.78751), (176.00866, 340.78751), (185.63362, 340.78751),
    (184.46401, 340.78751), (194.08896, 340.78751), (192.91935, 340.78751),
    (210.98332, 340.78751), (209.81371, 340.78751), (219.43866, 340.78751),
    (218.26905, 340.78751), (227.89400, 340.78751), (226.72439, 340.78751),
    (236.34934, 340.78751), (235.17973, 340.78751), (243.60639, 340.78751),
    (252.80811, 350.85255), (252.81298, 344.09633), (252.80811, 338.67191),
]

# All three levels: Floor 1 <-> 2 and Floor 2 <-> 3  (pairs (0, 1) and (1, 2))
LEVEL_ALL = [
    (241.12391, 352.16618),
    (196.67613, 352.44900),
    (141.61199, 352.16760),
]

# Build STAIR_LOCATIONS in the (X, Y, [(floor_a, floor_b), ...]) format.
STAIR_LOCATIONS = (
    [(x, y, [(1, 2)])         for (x, y) in LEVEL_3_TO_2]
    + [(x, y, [(0, 1)])         for (x, y) in LEVEL_2_TO_1]
    + [(x, y, [(0, 1), (1, 2)]) for (x, y) in LEVEL_ALL]
)
print(f"{len(STAIR_LOCATIONS)} stair locations "
      f"({len(LEVEL_3_TO_2)} for 3<->2, {len(LEVEL_2_TO_1)} for 2<->1, {len(LEVEL_ALL)} for all)")

SAVE_IMAGES = True

def save_fig(fig, filename):
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1800, height=1100, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

* `extract_triangles` / `points_inside` — geometry helpers (pull triangles, point-in-mesh test).
* `find_closest_node` — the instructor's `find_closest_vertex`, snaps a stair location to the nearest grid node.
* `make_cell_face` + `show_face_heatmap` — turn each grid node into a filled square cell and render it as a `Topology.Show` heatmap (S03 export format).

In [ ]:
def extract_triangles(face_list):
    # Return (T,3,2) array of plan-space (X,Y) triangles for a list of topologic faces.
    tris = []
    for f in face_list:
        vs = Topology.Vertices(f)
        pts = [(Vertex.X(v), Vertex.Y(v)) for v in vs]
        for i in range(1, len(pts) - 1):          # fan-triangulate (faces are already triangles)
            tris.append([pts[0], pts[i], pts[i + 1]])
    return np.array(tris)

def points_inside(tris, P):
    # Boolean mask: which points in P (N,2) fall inside ANY triangle of tris (T,3,2).
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    v0 = b - a; v1 = c - a
    d00 = (v0 * v0).sum(1); d01 = (v0 * v1).sum(1); d11 = (v1 * v1).sum(1)
    den = d00 * d11 - d01 * d01
    den[den == 0] = 1e-12
    inside = np.zeros(len(P), bool)
    for i, p in enumerate(P):
        v2 = p - a
        d20 = (v2 * v0).sum(1); d21 = (v2 * v1).sum(1)
        u = (d11 * d20 - d01 * d21) / den
        w = (d00 * d21 - d01 * d20) / den
        if np.any((u >= -1e-6) & (w >= -1e-6) & (u + w <= 1 + 1e-6)):
            inside[i] = True
    return inside

def find_closest_node(node_xy, x, y):
    # Index of the grid node closest to (x, y) -- the instructor's find_closest_vertex.
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, h):
    # A flat square cell of half-size h centred at (cx, cy), z = 0.
    pts = [Vertex.ByCoordinates(cx - h, cy - h, 0.0), Vertex.ByCoordinates(cx + h, cy - h, 0.0),
           Vertex.ByCoordinates(cx + h, cy + h, 0.0), Vertex.ByCoordinates(cx - h, cy + h, 0.0)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

def show_face_heatmap(faces_values, title, filename, colorScale="viridis"):
    # faces_values: list of (face, value). Colours each cell and renders the filled
    # heatmap with Topology.Show, exactly like the S03 notebooks (faceColorKey, black bg).
    vals = [v for _, v in faces_values]
    mn, mx = float(min(vals)), float(max(vals))
    if mx == mn: mx = mn + 1e-9
    for f, val in faces_values:
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Topology.Dictionary(f)
        d = Dictionary.SetValueAtKey(d, "hm_color", col)
        Topology.SetDictionary(f, d)
    faces = [f for f, _ in faces_values]
    fig = Topology.Show(faces, faceColorKey="hm_color", faceOpacity=1.0,
                        showEdges=False, showVertices=False, camera=[0, 0, 6],
                        backgroundColor="black", width=1700, height=1000,
                        showFigure=False, renderer=renderer)
    for fi in range(len(FLOOR_LEVELS)):
        yc = (fi - 1) * ROWGAP
        fig.add_trace(go.Scatter3d(x=[-(UMAX - UMIN) / 2 - 7], y=[yc], z=[0], mode="text",
                                   text=[FLOOR_NAMES[fi]], textfont=dict(color="white", size=16),
                                   showlegend=False))
    fig.update_layout(title=dict(text=title, font=dict(color="white")))
    fig.show(renderer=renderer)
    save_fig(fig, filename)
    return fig

## 5. Import the OBJ and split it into the three floor plans

`Topology.ByOBJPath` returns clusters of triangulated faces. We collect every face and bin it by its centroid's Z value into the three floor levels, then compute the shared plan bounding box.

In [ ]:
result = Topology.ByOBJPath(OBJ_PATH)
all_faces = []
for item in result:
    if Topology.IsInstance(item, "Cluster"):
        cf = Cluster.Faces(item)
        if cf: all_faces.extend(cf)
    elif Topology.IsInstance(item, "Face"):
        all_faces.append(item)
print(f"Imported {len(all_faces)} triangulated faces")

floor_faces = {lv: [] for lv in FLOOR_LEVELS}
for f in all_faces:
    z = Vertex.Z(Topology.Centroid(f))
    lv = min(FLOOR_LEVELS, key=lambda k: abs(k - z))
    floor_faces[lv].append(f)
for i, lv in enumerate(FLOOR_LEVELS):
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_faces[lv])} faces")

# Shared plan bounding box + vertical row gap used to stack floors in the 2D heatmaps
allxy = np.vstack([extract_triangles(floor_faces[lv]).reshape(-1, 2) for lv in FLOOR_LEVELS])
UMIN, VMIN = allxy.min(0)
UMAX, VMAX = allxy.max(0)
UMID, VMID = 0.5 * (UMIN + UMAX), 0.5 * (VMIN + VMAX)
ROWGAP = (VMAX - VMIN) + 8.0
print(f"Plan box: u[{UMIN:.1f},{UMAX:.1f}]  v[{VMIN:.1f},{VMAX:.1f}]")

## 6. Show the three raw floor plans

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    for t in tris:
        xs = list(t[:, 0]) + [t[0, 0]]
        ys = list(t[:, 1]) + [t[0, 1]]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                                 line=dict(color="rgba(170,195,255,0.45)", width=0.4),
                                 fillcolor="rgba(70,120,235,0.35)", showlegend=False), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Three imported floor plans (plan view)", font=dict(color="white")),
                  height=300 * len(FLOOR_LEVELS), width=1500,
                  paper_bgcolor="black", plot_bgcolor="black")
fig.show(renderer=renderer)
save_fig(fig, "01_floor_plans.png")

## 7. Sample a navigable grid on each floor

A regular grid is laid over the shared bounding box; only points inside the meshed (navigable) area are kept. These become the graph nodes of each floor.

In [ ]:
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])

floor_valid = {}    # level -> valid_xy ndarray
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    floor_valid[lv] = GRID_PTS[points_inside(tris, GRID_PTS)]
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_valid[lv])} navigable nodes")

## 8. Show the navigable grids (use these coordinates to locate your stairs)

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    fig.add_trace(go.Scatter(x=valid[:, 0], y=valid[:, 1], mode="markers",
                             marker=dict(size=5, color="royalblue"), showlegend=False), row=i + 1, col=1)
    sx = [s[0] for s in STAIR_LOCATIONS]; sy = [s[1] for s in STAIR_LOCATIONS]
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="markers",
                             marker=dict(size=14, color="red", symbol="x"), name="stairs",
                             showlegend=(i == 0)), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Navigable grids + stair locations (red x)", font=dict(color="white")),
                  height=300 * len(FLOOR_LEVELS), width=1500,
                  paper_bgcolor="black", plot_bgcolor="black",
                  legend=dict(font=dict(color="white")))
fig.show(renderer=renderer)
save_fig(fig, "02_navigable_grids.png")

## 9. Build the per-floor graphs, the display cells, and stack everything

For each floor the valid points become topologic vertices at `Z = floor_index * FLOOR_HEIGHT` (for the 3D graph) and a flat square **display cell** centred at `(u, v + offset)` (for the 2D heatmaps). Horizontal edges join 4-neighbour valid points.

In [ ]:
all_v = []            # topologic vertices (all floors, stacked in Z)
all_e = []            # topologic edges
floor_index_map = {}  # level -> {(round u, round v): global vertex index}
display_faces = []    # flat square cells for the heatmaps
cell_lookup = {}      # (floor_index, (round u, round v)) -> display face
H = GRID_SIZE / 2.0

for fi, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    z = fi * FLOOR_HEIGHT
    yoff = (fi - 1) * ROWGAP
    idx = {}
    for (u, v) in valid:
        idx[rk(u, v)] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z)))
        cell = make_cell_face(float(u) - UMID, (float(v) - VMID) + yoff, H)
        display_faces.append(cell)
        cell_lookup[(fi, rk(u, v))] = cell
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f"  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges")
print(f"Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges, {len(display_faces)} display cells")

def heatmap_from_graph(values, title, filename, colorScale):
    # Pair every graph vertex with its display cell, then render the filled heatmap.
    fv = []
    for v, val in zip(gverts, values):
        fi = int(round(Vertex.Z(v) / FLOOR_HEIGHT))
        f = cell_lookup.get((fi, rk(Vertex.X(v), Vertex.Y(v))))
        if f is not None:
            fv.append((f, val))
    return show_face_heatmap(fv, title, filename, colorScale)

## 10. Connect the floors through the stairs

For every stair location and adjacent floor pair we snap to the closest navigable node on each floor (`find_closest_node`) and add a **vertical stair edge** — this turns three separate plans into one connected building.

In [ ]:
stair_node_pairs = []
for stair in STAIR_LOCATIONS:
    sx, sy = stair[0], stair[1]
    # Use the floor pairs declared for this stair; otherwise connect every adjacent floor.
    pairs = stair[2] if len(stair) >= 3 and stair[2] else [(i, i + 1) for i in range(len(FLOOR_LEVELS) - 1)]
    for fa, fb in pairs:
        a, b = FLOOR_LEVELS[fa], FLOOR_LEVELS[fb]
        va, vb = floor_valid[a], floor_valid[b]
        ia = find_closest_node(va, sx, sy)
        ib = find_closest_node(vb, sx, sy)
        gia = floor_index_map[a][rk(va[ia, 0], va[ia, 1])]
        gib = floor_index_map[b][rk(vb[ib, 0], vb[ib, 1])]
        all_e.append(Edge.ByVertices([all_v[gia], all_v[gib]]))
        stair_node_pairs.append((gia, gib))
print(f"Added {len(stair_node_pairs)} vertical stair edges "
      f"from {len(STAIR_LOCATIONS)} stair location(s)")

## 11. Build the combined BUILDING graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f"Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)")
print(f"Graph density:  {Graph.Density(building_graph):.5f}")